In [2]:
import cv2 as cv
import torch
from ultralytics import YOLO
from collections import deque
import time
import numpy as np

In [3]:
def mouse_click(event, x,y, flags,param):
    if event==cv.EVENT_LBUTTONDOWN:
        print( f"({x}, {y})")


In [4]:
print(torch.cuda.is_available())


False


In [5]:
model=YOLO("yolo11n.pt")
cap=cv.VideoCapture("../data/3min.mp4")


In [6]:
if not cap.isOpened():
    print("Не удалось открыть видео — проверьте путь или кодек")

In [7]:
colors = {
    "car": (0, 255, 0),
    "bus": (255, 0, 0),
}

In [8]:
#quick remove last item (pop(0))
fps_history=deque(maxlen=30)

In [9]:
x1_roi,y1_roi,x2_roi,y2_roi=250,600,720,850

x: 457, y: 766
x: 601, y: 707
x: 716, y: 757
x: 626, y: 833

x: 286, y: 689
x: 431, y: 646
x: 562, y: 697
x: 426, y: 750

In [17]:
zones_points=[[(460,770),(600,700),(720,760),(630,840)], \
              [(300,700),(430,650),(560,700),(430,750)]]

In [31]:
zone_before=np.array(zones_points[0],dtype=np.int32)
zone_after=np.array(zones_points[1],dtype=np.int32)


In [34]:
frame

array([[[53, 23, 15],
        [52, 22, 14],
        [51, 23, 15],
        ...,
        [46, 10, 26],
        [42, 15, 20],
        [41, 14, 19]],

       [[52, 22, 14],
        [54, 24, 16],
        [51, 23, 15],
        ...,
        [41,  5, 21],
        [39, 12, 17],
        [38, 11, 16]],

       [[50, 22, 14],
        [52, 24, 16],
        [52, 24, 16],
        ...,
        [33,  9, 18],
        [37,  8, 18],
        [40, 11, 21]],

       ...,

       [[44, 47, 60],
        [43, 46, 59],
        [43, 46, 57],
        ...,
        [ 8,  3,  4],
        [ 9,  4,  5],
        [ 9,  4,  5]],

       [[44, 47, 60],
        [43, 46, 59],
        [44, 47, 58],
        ...,
        [ 8,  3,  4],
        [ 9,  4,  5],
        [ 9,  4,  5]],

       [[43, 46, 59],
        [43, 46, 59],
        [43, 46, 57],
        ...,
        [ 8,  3,  4],
        [ 8,  3,  4],
        [ 8,  3,  4]]], shape=(1280, 720, 3), dtype=uint8)

In [45]:
zone_before

array([[460, 770],
       [600, 700],
       [720, 760],
       [630, 840]], dtype=int32)

In [42]:
def add_layer(img, points, mask_color=(0, 255, 255), alpha=0.3):
    bin_mask = np.zeros((img.shape[:2]), dtype=np.uint8)
    bin_mask = cv.fillPoly(bin_mask, pts=[points], color=1)
    colored_mask = (bin_mask[:, :, np.newaxis] * mask_color).astype(np.uint8)
    return cv.addWeighted(img, 1, colored_mask, alpha, 0)

In [48]:
cap=cv.VideoCapture("../data/3min.mp4")
cv.namedWindow("frame")
cv.setMouseCallback("frame", mouse_click)
prev_time=time.time()
back_layer=None
while cap.isOpened():
    ret, frame= cap.read()
    if back_layer is None:
        back_layer=add_layer(np.zeros((frame.shape), dtype=np.uint8),zone_before,mask_color=(0, 255, 0))
        back_layer=add_layer(back_layer,zone_after,mask_color=(0, 0, 255))
    # results = model.predict(frame[y1_roi:y2_roi,x1_roi:x2_roi],classes=[2,3,5,7], verbose=False)[0]
    results = model.track(frame[y1_roi:y2_roi,x1_roi:x2_roi],persist=True, classes=[2,3,5,7], verbose=False)[0]
    # zones=frame.copy()
    # cv.fillPoly(zones,[zone_before],color=(0, 255, 0))
    # cv.fillPoly(zones,[zone_after],color=(0, 0, 255))
    # frame=cv.addWeighted(zones,0.3, frame, 0.7, 0)
    frame=cv.addWeighted(back_layer,1, frame, 1, 0)
    cur_time=time.time()
    fps_history.append(1/(cur_time-prev_time))
    prev_time=time.time()
    avg_fps=sum(fps_history)/len(fps_history)
    cv.putText(frame, f"FPS: {avg_fps:.1f}",(10,30), cv.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

    for result in results.boxes:
        x1,y1, x2,y2=map(int,result.xyxy[0])
        #transform coord from roi_frame to origin frame
        x1, x2 = x1 + x1_roi, x2 + x1_roi
        y1, y2 = y1 + y1_roi, y2 + y1_roi
        
        cv.rectangle(frame,(x1,y1),(x2,y2),(255,0,0),3)
        class_id=int(result.cls[0])
        class_name=model.names[class_id]
        conf=result.conf[0]
        color_ob=colors.get(class_name,(255,255,255))
        track_id = int(result.id[0]) if result.id is not None else -1
        ob_label=f"{class_name} #{track_id}  {conf:.2f}"
        

        cv.putText(frame, ob_label,(x1,y1-8), cv.FONT_HERSHEY_SIMPLEX, 0.5, color_ob, 2)
        cv.imshow("frame",frame)
    if cv.waitKey(1)==ord("q"):
        break
print(avg_fps)
cap.release()
cv.destroyAllWindows()


(440, 355)
(436, 317)
17.323703278752046


In [37]:
print(results.speed)

{'preprocess': 1.1560000020836014, 'inference': 34.292999996978324, 'postprocess': 0.8593999991717283}


In [30]:
import torch
print(torch.get_num_threads())

8


In [ ]:
results = model.predict(source="../data/3min.mp4", save=True, stream=True)

In [ ]:
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    xywhn = result.boxes.xywhn  # normalized
    xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
    xyxyn = result.boxes.xyxyn  # normalized
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box

In [ ]:
model.names